# Milestone 1 demo: load saved weights and predict

Weights: `models/qwen2.5-1.5b-disaster-lora/` (adapter only; the 1.5B base stays in the Hugging Face cache). Training lives in `eda_and_baseline.ipynb`.

In [1]:
from pathlib import Path
import json, os, warnings

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from peft import AutoPeftModelForSequenceClassification
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

warnings.filterwarnings("ignore")
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

ROOT = Path.cwd()
if not (ROOT / "data" / "train.csv").exists():
    ROOT = Path("/Users/nine/Desktop/CMKL/3.1/sys304/milestone/1")
DATA = ROOT / "data"
SAVE_DIR = ROOT / "models" / "qwen2.5-1.5b-disaster-lora"
MAX_LEN = 128
LABELS = {0: "not disaster", 1: "disaster"}

adapter = SAVE_DIR / "adapter_model.safetensors"
assert adapter.exists(), f"missing {adapter} — run eda_and_baseline.ipynb first"
metrics = json.loads((SAVE_DIR / "metrics.json").read_text())

print("adapter:", adapter, f"({adapter.stat().st_size / 1e6:.1f} MB)")
print("saved validation metrics (from training, 1,523 tweets):")
print(json.dumps(metrics, indent=2))

adapter: /Users/nine/Desktop/CMKL/3.1/sys304/milestone/1/models/qwen2.5-1.5b-disaster-lora/adapter_model.safetensors (17.5 MB)
saved validation metrics (from training, 1,523 tweets):
{
  "model_id": "Qwen/Qwen2.5-1.5B-Instruct",
  "val_f1_disaster": 0.816260162601626,
  "val_f1_weighted": 0.8500903943339057,
  "tfidf_f1_disaster": 0.7773527161438408,
  "n_train": 6090,
  "n_val": 1523
}


## 1. Load tweets

Same input format as training: `keyword: …` on the first line, then the tweet body.

In [2]:
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
train["keyword"] = train["keyword"].fillna("").str.replace("%20", " ", regex=False)
test["keyword"] = test["keyword"].fillna("").str.replace("%20", " ", regex=False)

def build_input(df: pd.DataFrame) -> pd.Series:
    kw = df["keyword"].fillna("").str.strip()
    text = np.where(kw.eq(""), df["text"], "keyword: " + kw + "\n" + df["text"])
    return pd.Series(text, index=df.index, dtype="string")

X = build_input(train)
y = train["target"].astype(int).to_numpy()
_, X_va, _, y_va = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"train {len(train):,}  test {len(test):,}  val holdout {len(X_va):,}")
display(train[["keyword", "text", "target"]].head(3))

train 7,613  test 3,263  val holdout 1,523


,keyword,text,target
0,,Our Deeds are the Reason of this #earthquake M...,1
1,,Forest fire near La Ronge Sask. Canada,1
2,,All residents asked to 'shelter in place' are ...,1


## 2. Load the trained LoRA adapter

This cell loads the 1.5B base from cache and attaches the saved adapter. It is the slow step (~30–60 s). Nothing is trained.

You will see `score.weight | MISSING` from transformers. That is the **base** Qwen causal-LM checkpoint (it has no classification head). The trained 2-class `score` head is in `adapter_model.safetensors` and is applied next.

In [3]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoPeftModelForSequenceClassification.from_pretrained(
    str(SAVE_DIR),
    dtype="float32",
)
model.config.pad_token_id = tokenizer.pad_token_id
model.to(device)
model.eval()
print("device:", device)
print("base:", metrics["model_id"])
print("loaded from", SAVE_DIR)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-1.5B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


device: mps
base: Qwen/Qwen2.5-1.5B-Instruct
loaded from /Users/nine/Desktop/CMKL/3.1/sys304/milestone/1/models/qwen2.5-1.5b-disaster-lora


## 3. Predictions on labeled validation tweets

Twelve tweets from the same 80/20 split used in training (not used to train). `true` is the Kaggle label; `pred` is the loaded model.

A 12-row slice will not match the official 1,523-tweet F1. Use the saved metrics for the real score; this table is for the recording.

In [4]:
@torch.inference_mode()
def predict_texts(texts: list[str], batch_size: int = 8):
    preds, confs = [], []
    for i in range(0, len(texts), batch_size):
        batch = list(texts[i : i + batch_size])
        enc = tokenizer(
            batch, truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt"
        )
        enc = {k: v.to(device) for k, v in enc.items()}
        logits = model(**enc, return_dict=True).logits
        prob = torch.softmax(logits, dim=-1)
        preds.append(logits.argmax(dim=-1).cpu().numpy())
        confs.append(prob.cpu().numpy())
    return np.concatenate(preds), np.concatenate(confs)

rng = np.random.default_rng(42)
texts = np.asarray(X_va)
true = np.asarray(y_va)
va = pd.DataFrame({"text": texts, "true": true})
pick = np.concatenate([
    rng.choice(np.flatnonzero(true == 1), size=6, replace=False),
    rng.choice(np.flatnonzero(true == 0), size=6, replace=False),
])
sample = va.iloc[pick].reset_index(drop=True)
pred, conf = predict_texts(sample["text"].tolist())
sample["pred"] = pred.astype(int)
sample["confidence"] = np.round(conf[np.arange(len(sample)), pred], 3)
sample["true_label"] = [LABELS[int(t)] for t in np.asarray(sample["true"])]
sample["pred_label"] = [LABELS[int(p)] for p in pred]
sample["ok"] = np.where(sample["true"] == sample["pred"], "correct", "wrong")

print(f"this 12-tweet slice: {(sample['true'] == sample['pred']).sum()}/12 correct  (demo only, not the official metric)")
print(f"official val F1 (disaster): {metrics['val_f1_disaster']:.3f}  weighted F1: {metrics['val_f1_weighted']:.3f}  on {metrics['n_val']} tweets")
display(sample[["text", "true_label", "pred_label", "confidence", "ok"]])

this 12-tweet slice: 9/12 correct  (demo only, not the official metric)
official val F1 (disaster): 0.816  weighted F1: 0.850  on 1523 tweets


,text,true_label,pred_label,confidence,ok
0,keyword: mudslide\nOso Washington Mudslide Res...,disaster,disaster,0.992,correct
1,keyword: hailstorm\nReady for my close up... E...,disaster,disaster,0.785,correct
2,keyword: explode\nKendall Jenner and Nick Jona...,disaster,not disaster,0.962,wrong
3,keyword: collided\nMonsoon flooding - Monsoon ...,disaster,disaster,0.999,correct
4,keyword: debris\nConfirmed the debris from MH3...,disaster,disaster,0.957,correct
5,keyword: bleeding\nyou can stab me in the back...,disaster,not disaster,0.919,wrong
6,keyword: fatality\n@FaTality_US need a team? W...,not disaster,not disaster,0.955,correct
7,keyword: deluge\nWA smiles after July deluge -...,not disaster,disaster,0.881,wrong
8,keyword: fatal\nRoger Goodell's Fatal Mistake:...,not disaster,not disaster,0.879,correct
9,keyword: bloody\n@zhenghxn i tried 11 eyes aka...,not disaster,not disaster,0.909,correct


## 4. Live examples

Handmade tweets so the recording shows obvious disaster vs metaphor cases.

In [5]:
live = pd.DataFrame({
    "text": [
        "keyword: earthquake\nMagnitude 6.5 quake hits Nepal, dozens trapped under rubble",
        "keyword: disaster\nThis movie is a disaster lol the acting is so bad",
        "keyword: forest fire\nForest fire near Yosemite is spreading, evacuations underway",
        "keyword: sinking\nI got a sinking feeling about this exam",
        "keyword: flood\nFlash flood warning issued for downtown after the dam broke",
        "Our BBQ is on fire and the burgers are actually perfect",
    ]
})
pred, conf = predict_texts(live["text"].tolist())
live["pred"] = pred.astype(int)
live["pred_label"] = [LABELS[int(p)] for p in pred]
live["confidence"] = np.round(conf[np.arange(len(live)), pred], 3)
display(live)

,text,pred,pred_label,confidence
0,keyword: earthquake\nMagnitude 6.5 quake hits ...,1,disaster,1.000
1,keyword: disaster\nThis movie is a disaster lo...,0,not disaster,0.916
2,keyword: forest fire\nForest fire near Yosemit...,1,disaster,1.000
3,keyword: sinking\nI got a sinking feeling abou...,0,not disaster,0.960
4,keyword: flood\nFlash flood warning issued for...,1,disaster,0.995
5,Our BBQ is on fire and the burgers are actuall...,0,not disaster,0.826


## 5. Unlabeled Kaggle test tweets

These have no `target`. Same format as `submission.csv`.

In [6]:
test_sample = test.head(8).copy()
texts = build_input(test_sample).tolist()
pred, conf = predict_texts(texts)
out = pd.DataFrame({
    "id": test_sample["id"].to_numpy(),
    "keyword": test_sample["keyword"].to_numpy(),
    "text": test_sample["text"].to_numpy(),
    "target": pred.astype(int),
    "pred_label": [LABELS[int(p)] for p in pred],
    "confidence": np.round(conf[np.arange(len(pred)), pred], 3),
})
display(out)
print("done — weights loaded from disk, predictions printed above")

,id,keyword,text,target,pred_label,confidence
0,0,,Just happened a terrible car crash,1,disaster,0.831
1,2,,"Heard about #earthquake is different cities, s...",1,disaster,0.969
2,3,,"there is a forest fire at spot pond, geese are...",1,disaster,0.855
3,9,,Apocalypse lighting. #Spokane #wildfires,1,disaster,0.966
4,11,,Typhoon Soudelor kills 28 in China and Taiwan,1,disaster,1.000
5,12,,We're shaking...It's an earthquake,0,not disaster,0.515
6,21,,They'd probably still show more life than Arse...,0,not disaster,0.863
7,22,,Hey! How are you?,0,not disaster,0.962


done — weights loaded from disk, predictions printed above
